<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/03%20-%20Unsupervised%20Learning/03a_dimensionality_reduction.ipynb)

# 03a — 降维：把高维化学空间画在二维平面上（Dimensionality Reduction）

在第二讲（`02 - Supervised Learning`）中，每个分子都带有明确的实验测量标签 $\log S$，模型学习的任务是从分子特征 $X$ 映射到性质 $y$。

但在真实的材料与化学研发中，我们经常先面对一个**没有实验标签**（或者只有极少数散落标签）的庞大候选分子库。此时我们首先要回答的是另一类问题：

| 典型科研问题 | 对应的无监督学习方法 | 本讲模块 |
|---|---|---|
| 这上万个分子在“化学空间”里是怎么分布的？有没有明显的家族聚集、数据偏倚或异常离群点？ | **降维与可视化（Dimensionality Reduction）** | `03a` |
| 能不能不靠人工规则，让算法自动把结构相似的分子归成若干个家族？ | **聚类（Clustering）** | `03b` |
| 手头只有 4 个已知有效的催化剂配体，无法训练监督模型，怎样从 300 多个候选配体里筛出下一批最值得合成的名单？ | **共识聚类与化学空间近邻筛选** | `03c` |

这类不依赖标签 $y$、直接从特征矩阵 $X$ 中挖掘数据内在几何结构与分布规律的方法，统称为**无监督学习（Unsupervised Learning）**。

本节（`03a`）聚焦于**降维（Dimensionality Reduction）**：当每个分子由 17 个物理化学描述符（17 维空间）或 2048 位 Morgan 指纹（2048 维空间）表示时，人类无法直接观察超过三维的空间。降维算法的目标，就是在二维平面上找到一个“信息损失最小”的投影，让我们能直观地看到整片化学空间的地形图。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "03 - Unsupervised Learning"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


---

## 0. 环境准备与 AqSolDB 数据集介绍

前两讲使用的 ESOL 数据集包含 1,128 个有机小分子。为了展示无监督学习在更大、更复杂分子库上的威力，本讲切换到 **AqSolDB**（Aqueous Solubility Database，Sorkun et al., *Sci. Data* 2019）：它整合了近万个化合物（包括有机小分子、药物、盐类、表面活性剂及工业化学品）的水溶解度数据，并预先计算好了 17 个 RDKit 分子描述符。

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw, rdFingerprintGenerator
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.manifold import TSNE

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# 本地运行默认读取 data/solubility.csv；在 Colab 上若找不到文件会提示上传
DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请确保它位于当前目录或 data/ 文件夹下")

df = pd.read_csv(DATA_FILE)
df["mol"] = df["SMILES"].apply(Chem.MolFromSmiles)
df = df[df["mol"].notna()].reset_index(drop=True)

feature_cols = [
    "MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors",
    "NumHeteroatoms", "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings",
    "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
    "BalabanJ", "BertzCT",
]
print(f"成功解析分子数: {len(df)} | 预置物理化学描述符数: {len(feature_cols)}")

---

## 1. 降维的两种核心哲学：保留全局方差 vs 保留局部近邻

把高维空间（如 17 维或 2048 维）拍扁到二维平面，必然会丢失一部分几何信息。不同的降维算法对于“优先保留什么信息”有着截然不同的数学目标：

1. **主成分分析（PCA, Principal Component Analysis）**：**线性投影**，致力于保留全局**方差（Variance）**最大的方向——即原本在空间里散得最开的维度，投影到平面后依然散得最开；
2. **t-SNE / UMAP**：**非线性流形降维**，不追求保持远距离样本之间的全局比例，而是致力于保留**局部近邻关系（Neighborhood）**——即高维空间里彼此靠近的“邻居”分子，在二维图上也要聚成一团。

---

## 2. 主成分分析（PCA）的几何直觉：从两个共线描述符说起

为了看清 PCA 在做什么，我们先只取两个大家最熟悉的描述符：重原子数 `HeavyAtomCount` 和分子量 `MolWt`。

在第一讲 `01c` 和第二讲 `02c` 中我们已经注意到，这两个特征高度正相关（重原子越多，分子量自然越大）。如果把它们先做 `StandardScaler` 标准化（消除量纲差异，使均值为 0、标准差为 1），再画在二维散点图上，数据点会沿着对角线方向拉成一条狭长的椭圆带：

In [ ]:
two = StandardScaler().fit_transform(df[["HeavyAtomCount", "MolWt"]])
pca2 = PCA(n_components=2).fit(two)

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.scatter(two[:, 0], two[:, 1], s=3, alpha=0.3)
for length, vector in zip(pca2.explained_variance_, pca2.components_):
    arrow = vector * 2 * np.sqrt(length)   # 箭头长度正比于该主成分方向上的标准差
    ax.annotate("", xy=arrow, xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=2.5, color="tab:red"))
ax.set_xlim(-3, 8)
ax.set_ylim(-3, 8)
ax.set_xlabel("HeavyAtomCount (standardized)")
ax.set_ylabel("MolWt (standardized)")
ax.set_title("Two correlated descriptors and their principal axes")
plt.tight_layout()
plt.show()

print("两个主成分各自解释的方差比例:", pca2.explained_variance_ratio_.round(3))

PCA 所做的事情，就是把原来的坐标轴旋转到一个新的正交坐标系（图中的两个红色箭头）：
- **第一主成分（PC1）**：沿着数据点最“散开”（方差最大）的对角线方向，单独一个方向就解释了 **96.5%** 的方差，它的物理意义就是**“分子的整体尺寸”**；
- **第二主成分（PC2）**：与 PC1 严格垂直，只占 **3.5%** 的方差，反映的是**“在重原子数相同的情况下，分子量偏大还是偏小”**（例如含有 Br、I 等重卤素原子的分子会偏离对角线）。

如果我们只保留 `PC1` 而丢弃 `PC2`，就把二维数据压缩成了一维，同时保留了 96.5% 的原始方差信息。**这就是 PCA 降维的本质：通过正交旋转消除特征间的线性冗余，只保留方差最大的前几个主轴。**

> **为什么 PCA 前必须做标准化？**
> PCA 寻找的是数值方差最大的方向。如果不做 `StandardScaler`，`MolWt` 的数值范围是几百（方差上万），而 `NumHDonors` 的数值只有 `0 ~ 5`（方差约为 1），PCA 算出的 PC1 就会几乎 100% 等于 `MolWt`。因此，对不同量纲的物理化学描述符做 PCA 前，必须先标准化。

---

## 3. 对 17 个物理化学描述符做 PCA：方差、载荷与化学意义

### 3.1 解释方差比（Explained Variance Ratio）：17 个描述符里有多少独立信息？

现在我们把全部 17 个标准化后的描述符输入 `PCA()`，看看每个主成分各自承担了多少方差，以及需要几个主成分就能累计保留 90% 的信息：

In [ ]:
scaler = StandardScaler().fit(df[feature_cols])
X_scaled = scaler.transform(df[feature_cols])

pca = PCA().fit(X_scaled)   # 不指定 n_components 时保留全部 17 个主成分
ratio = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.bar(range(1, 18), ratio, label="each PC")
ax.plot(range(1, 18), np.cumsum(ratio), "o-", color="tab:red", label="cumulative")
ax.axhline(0.9, color="gray", ls=":", label="90% threshold")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance Ratio")
ax.set_title("PCA on 17 RDKit Descriptors")
ax.legend()
plt.tight_layout()
plt.show()

print("前 5 个主成分各自的解释方差比:", ratio[:5].round(3))
print("前 2 个主成分累计解释方差比 :", ratio[:2].sum().round(3))
print("前 5 个主成分累计解释方差比 :", ratio[:5].sum().round(3))

结果非常清晰：
- **PC1 单独就占了 54.0% 的方差**，前两个主成分（PC1 + PC2）合计占 **68.4%**，**前 5 个主成分累计就超过了 90%**！
- 这从定量角度印证了我们在第二讲看到的共线性现象：表面上看我们算了 17 个不同的描述符，但其中 `MolWt`、`HeavyAtomCount`、`MolMR`（摩尔折射率）、`LabuteASA`（近似表面积）、`NumValenceElectrons`（价电子数）高度同源，本质上都在重复描述“分子体积/大小”。

### 3.2 主成分载荷（Loadings）：每个主成分代表什么物理化学属性？

每个主成分都是原始 17 个标准化描述符的线性组合：
$$\text{PC}_k = w_{k,1} \tilde{x}_1 + w_{k,2} \tilde{x}_2 + \dots + w_{k,17} \tilde{x}_{17}$$
组合系数 $w_{k,j}$ 存储在 `pca.components_` 中，称为**载荷（Loadings）**。画出前三个主成分的载荷条形图，就能直接读出它们的物理化学含义：

In [ ]:
loadings = pd.DataFrame(pca.components_[:3].T, index=feature_cols, columns=["PC1", "PC2", "PC3"])

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
for ax, pc in zip(axes, loadings.columns):
    colors = np.where(loadings[pc] > 0, "tab:blue", "tab:orange")
    ax.barh(loadings.index, loadings[pc], color=colors)
    ax.axvline(0, color="gray", lw=0.6)
    ax.set_title(f"{pc} Loadings ({ratio[int(pc[-1])-1]:.0%} var)")
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

从载荷图上可以清楚地总结出前三个正交主轴的化学身份：
1. **PC1（54% 方差）—— 分子尺寸（Molecular Size）**：几乎所有与原子数、分子量、电子数、表面积相关的描述符都具有一致的正载荷；
2. **PC2（14% 方差）—— 疏水/亲油 vs 极性/亲水（Lipophilicity vs Polarity）**：正向载荷最大的是 `MolLogP` 和 `NumRotatableBonds`（长疏水链），负向载荷最大的是极性表面积 `TPSA`、氢键受体/给体数（`NumHAcceptors`、`NumHDonors`）与杂原子数（`NumHeteroatoms`）；
3. **PC3（9% 方差）—— 脂肪环/饱和环特征（Aliphatic Ring Character）**：正向载荷集中在 `NumSaturatedRings` 和 `NumAliphaticRings`（如甾体、环烷烃类骨架）。

### 3.3 绘制二维 PCA 化学空间地图并叠加溶解度

既然 `PC1` 代表分子大小、`PC2` 代表亲油性 vs 极性，我们把 9,480 个分子投影到 `(PC1, PC2)` 二维平面上，并用实验水溶解度 `Solubility`（$\log S$）给散点上色，看看会发生什么：

In [ ]:
pcs = pca.transform(X_scaled)
df["PC1"], df["PC2"] = pcs[:, 0], pcs[:, 1]

fig, ax = plt.subplots(figsize=(6.6, 5.2))
sc = ax.scatter(
    df["PC1"], df["PC2"],
    c=df["Solubility"], cmap="viridis", s=4, alpha=0.6, vmin=-10, vmax=2
)
plt.colorbar(sc, label="Measured logS")
ax.set_xlabel(f"PC1 ({ratio[0]:.0%}): Molecular Size")
ax.set_ylabel(f"PC2 ({ratio[1]:.0%}): Lipophilic (+) vs Polar (-)")
ax.set_title("AqSolDB in Descriptor PCA Space")
plt.tight_layout()
plt.show()

print("PC1 与 logS 的皮尔逊相关系数:", np.corrcoef(df["PC1"], df["Solubility"])[0, 1].round(2))
print("PC2 与 logS 的皮尔逊相关系数:", np.corrcoef(df["PC2"], df["Solubility"])[0, 1].round(2))

可以看到：散点图的颜色渐变主要沿着纵轴 **PC2（亲油 vs 极性）** 方向展开（$r = -0.49$），越往上（越亲油）颜色越深（越难溶）；而沿着横轴 **PC1（分子大小）** 的相关性则弱得多（$r = -0.28$）。

**这一点非常值得玩味**：PCA 是一个纯粹的无监督算法，它在计算主成分时**完全没有见过 `logS` 标签**；但它仅凭分子描述符自身的协方差结构提取出的第二主轴（亲油 vs 极性），恰恰对应了 Yalkowsky 溶解度方程中最核心的热力学驱动力！

当然，解读二维 PCA 图时也要保持谨慎：`PC1 + PC2` 合计只保留了 68% 的方差，还有 32% 的信息折叠在垂直于纸面的其余 15 个维度里，因此**平面上看起来紧挨着的两个点，在 17 维全空间里不一定真的完全相同**。

### 🏋️ 练习 03a-1：检视 PC2 两端的极端分子（兼谈降维用于异常数据排查）

用 `df.nlargest(5, "PC2")` 和 `df.nsmallest(5, "PC2")` 找出 `PC2` 最大（极度亲油）和最小（极度极性）的各 5 个分子，并用 `Draw.MolsToGridImage` 画出它们的结构式（图例标出 `Name` 和 `Solubility`）。观察：
1. 它们的结构符合“亲油 vs 极性”的化学解释吗？
2. 在 `PC2` 最大的一端，你能发现数据集中混入了什么“不太像常规有机小分子”的条目吗？

In [ ]:
# 在下方填写你的代码：


---

## 4. 投影新分子与定量适用域检查：有机半导体分子落在哪里？

### 4.1 用已拟合的 `scaler` 和 `pca` 投影新分子

PCA 学到的是一个固定的线性坐标变换矩阵。当我们拿到一批新的候选分子（例如前两讲反复出现的 9 个有机半导体结构单元与分子）时，**绝不能把新分子混进原数据集重新 `fit`**（否则坐标系本身会发生偏移），而是直接调用在 AqSolDB 上拟合好的 `scaler.transform()` 和 `pca.transform()`，将它们投射到同一张化学空间底图上：

In [ ]:
candidates = {
    "thiophene":        "c1ccsc1",
    "EDOT":             "C1COc2cscc2O1",
    "3-hexylthiophene": "CCCCCCc1ccsc1",
    "carbazole":        "c1ccc2c(c1)[nH]c1ccccc12",
    "triphenylamine":   "c1ccc(cc1)N(c1ccccc1)c1ccccc1",
    "pentacene":        "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "BTBT":             "c1ccc2c(c1)sc1c3ccccc3sc21",
    "C8-BTBT":          "CCCCCCCCc1ccc2c(c1)sc1c3ccc(CCCCCCCC)cc3sc21",
    "rubrene":          "c1ccc(cc1)-c1c2ccccc2c(-c2ccccc2)c2c(-c3ccccc3)c3ccccc3c(-c3ccccc3)c12",
}

def descriptor_row(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return [getattr(Descriptors, name)(mol) for name in feature_cols]

osc = pd.DataFrame(
    [descriptor_row(s) for s in candidates.values()],
    columns=feature_cols,
    index=list(candidates),
)
osc_pcs = pca.transform(scaler.transform(osc))   # 严格复用已有的 scaler 和 pca！

fig, ax = plt.subplots(figsize=(7.2, 5.5))
ax.scatter(df["PC1"], df["PC2"], s=3, color="lightgray", label="AqSolDB")
ax.scatter(osc_pcs[:, 0], osc_pcs[:, 1], s=45, color="tab:red", zorder=5, label="Organic Semiconductors")
for name, (x, y) in zip(candidates, osc_pcs[:, :2]):
    ax.annotate(name, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=9)
ax.set_xlim(-5, 15)
ax.set_ylim(-8, 8)   # 聚焦主分布区（极少数超大分子离群点在此范围外）
ax.set_xlabel("PC1: Molecular Size")
ax.set_ylabel("PC2: Lipophilic (+) vs Polar (-)")
ax.legend(loc="lower right")
ax.set_title("Where do Organic Semiconductors sit in AqSolDB PCA Space?")
plt.tight_layout()
plt.show()

### 4.2 在 17 维全空间中定量测量最近邻距离（适用域诊断）

前面提到，二维 PCA 图只保留了 68% 的方差，“图上看着在内部或边缘”可能受到维度折叠的误导。最严谨的做法是直接在 **17 维标准化特征空间**里用 `NearestNeighbors` 计算每个候选分子到 AqSolDB 最近邻分子的欧氏距离，并与 AqSolDB 内部点的最近邻距离分布（中位数与 95% 分位数）做对照：

In [ ]:
nn = NearestNeighbors(n_neighbors=2).fit(X_scaled)
d_inside = nn.kneighbors(X_scaled)[0][:, 1]   # 第 0 近邻是样本自身（距离为 0），取第 1 近邻
print(f"AqSolDB 内部最近邻距离 —— 中位数: {np.median(d_inside):.2f} | 95% 分位数: {np.percentile(d_inside, 95):.2f}")

d_osc, i_osc = nn.kneighbors(scaler.transform(osc), n_neighbors=1)
pd.DataFrame({
    "17D Distance to Nearest": d_osc[:, 0].round(2),
    "Nearest AqSolDB Molecule": df["Name"].values[i_osc[:, 0]],
}, index=list(candidates))

这张定量距离表揭示了三个非常有启发性的现象：
1. **常见砌块在适用域内**：`thiophene` 和 `carbazole` 本身就收录在 AqSolDB 中（距离 `0.00`）；`EDOT`、`3-hexylthiophene` 和 `triphenylamine` 离最近邻的距离都在 `0.5` 以内，低于数据集内部中位数（`0.39`），处于数据密集的主体区域。
2. **为什么并五苯（`pentacene`）的距离只有 `0.18`？** 它的最近邻是**二苯并[a,h]蒽（dibenz[a,h]anthracene）**——一个与并五苯分子式完全相同（$\text{C}_{22}\text{H}_{14}$，五个稠合苯环）的**同分异构体**！在 17 个全局拓扑/物理化学描述符看来，线性并五苯和弯曲的二苯并蒽几乎一模一样（仅拓扑指数 `BalabanJ` 有细微差别）。但正如我们在 `02b` 所见，线性并五苯的晶格堆积更紧密、溶解度比弯曲异构体低得多——这再次暴露了全局 2D 描述符对“同分异构体空间构型差异”的盲区。
3. **真正的域外大分子**：`C8-BTBT`（距离 `1.85`）和红荧烯 `rubrene`（距离 `2.19`）的最近邻距离远超 AqSolDB 内部 95% 分位数（`1.34`），在 PCA 图上也处于右上角极度稀疏的边缘地带。对这类既大又极度疏水的共轭分子直接套用模型预测，外推风险极高。

---

## 5. 对 2048 维 Morgan 指纹做 PCA：为什么线性降维会失效？

既然 17 个描述符用 5 个主成分就能概括 90% 的方差，那么如果我们把分子表征换成 2048 位的 **Morgan 二值指纹（Bit Fingerprint）**，再用 PCA 降维会怎样？

In [ ]:
fp_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = np.array([fp_gen.GetFingerprintAsNumPy(m) for m in df["mol"]], dtype=np.uint8)
print(f"指纹矩阵形状: {fps.shape} | 平均每个分子点亮的比特数: {fps.sum(axis=1).mean():.1f}")

pca_fp = PCA(n_components=50, random_state=42).fit(fps)
print("前 2 个主成分解释的方差比 :", pca_fp.explained_variance_ratio_[:2].round(3))
print("前 50 个主成分累计解释方差:", pca_fp.explained_variance_ratio_.sum().round(3))

前两个主成分加起来只解释了 **9.3%** 的方差，哪怕保留前 **50 个主成分**，累计解释方差也只有 **45.2%**（不到一半）！

**原因在于指纹的高维稀疏性与非线性独立性**：2048 位里平均每个分子只有约 22 位是 `1`，每一位对应一个极其具体的局部原子环境（例如“某个带氟的苯环片段”或“某个叔丁基片段”）。这些成百上千种局部子结构之间大多彼此独立，不存在像“分子量—重原子数—表面积”那样一条贯穿全局的线性主轴。面对这种高维稀疏数据，线性 PCA 无法在二维平面上有效展开化学空间。

---

## 6. t-SNE：保留“局部邻居”关系的非线性流形降维

### 6.1 t-SNE 的工作原理与标准预处理流水线

既然线性投影行不通，**t-SNE**（t-Distributed Stochastic Neighbor Embedding）换了一种思路：**放弃保持全局距离比例，专心让高维空间里互为近邻的分子在二维平面上也聚在一起。**

它的核心步骤分为三步：
1. 在高维空间中，根据点对之间的距离计算每个点选择其他点作为“邻居”的条件概率分布（离得越近概率呈高斯衰减越大，有效邻居数由超参数 `perplexity` 控制，常取 `5 ~ 50`）；
2. 在二维平面上用自由度为 1 的重尾 $t$ 分布定义低维邻居概率分布（重尾分布能有效缓解高维球壳往二维挤压时的“拥挤问题 Crowding Problem”）；
3. 通过梯度下降最小化高维与低维概率分布之间的 KL 散度（Kullback-Leibler Divergence），不断移动二维点位置直到邻居关系吻合。

> **工程加速惯例**：直接对 $9480 \times 2048$ 的稀疏矩阵算两两距离非常耗时且易受哈希碰撞噪声干扰。标准做法是**先用 `PCA(n_components=50)` 滤除高频噪声并压缩到 50 维，再输入 `TSNE`**（约需 30–60 秒）：

In [ ]:
fp_50 = pca_fp.transform(fps)
tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=42)
emb = tsne.fit_transform(fp_50)
df["tSNE1"], df["tSNE2"] = emb[:, 0], emb[:, 1]

为了看清 t-SNE 把哪些分子聚在了一起，我们先写一个简单的结构分类函数，把分子粗略分为 5 大结构类型（多组分盐/混合物、无环分子、纯脂肪环、含卤芳环、普通芳环），然后把按结构类型着色的 t-SNE 图与按 `logS` 着色的图并排对比：

In [ ]:
def rough_class(mol, smiles):
    if "." in smiles:
        return "salt / mixture"
    if mol.GetRingInfo().NumRings() == 0:
        return "acyclic"
    if Descriptors.NumAromaticRings(mol) == 0:
        return "aliphatic rings only"
    if any(a.GetSymbol() in ("F", "Cl", "Br", "I") for a in mol.GetAtoms()):
        return "aromatic + halogen"
    return "aromatic"

df["class"] = [rough_class(m, s) for m, s in zip(df["mol"], df["SMILES"])]
df["class"].value_counts()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for name, group in df.groupby("class"):
    axes[0].scatter(group["tSNE1"], group["tSNE2"], s=3, alpha=0.6, label=name)
axes[0].legend(markerscale=4, fontsize=8)
axes[0].set_title("t-SNE of Morgan Fingerprints (Colored by Structural Class)")

sc = axes[1].scatter(
    df["tSNE1"], df["tSNE2"],
    c=df["Solubility"], cmap="viridis", s=3, vmin=-10, vmax=2
)
plt.colorbar(sc, ax=axes[1], label="Measured logS")
axes[1].set_title("Same t-SNE Map (Colored by logS)")

for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

### 6.2 深入观察一个局部“岛屿”：当指纹相似遇上性质突变（Activity Cliff）

与指纹的线性 PCA 不同，t-SNE 把 9,480 个分子清晰地分成了许多**“结构岛屿”**：左图中同一结构大类的分子各自抱团，右图中大多数岛屿内部的颜色（溶解度 `logS`）也相当一致——这直观验证了**“结构相似的分子往往具有相近的物理化学性质”**。

但如果我们放大看某一个具体的岛屿（例如以第 `0` 号分子为中心，找出 t-SNE 图上离它最近的 8 个邻居），会发现一个非常有趣的化学现象：

In [ ]:
center = 0   # 可修改行号查看其他岛屿
d = np.hypot(df["tSNE1"] - df.loc[center, "tSNE1"], df["tSNE2"] - df.loc[center, "tSNE2"])
neighbours = df.loc[d.nsmallest(8).index]
Draw.MolsToGridImage(
    neighbours["mol"].tolist(),
    molsPerRow=4,
    subImgSize=(220, 180),
    legends=[f"logS = {s:.1f}" for s in neighbours["Solubility"]],
)

第 `0` 号分子是**十八烷基三甲基溴化铵**（阳离子表面活性剂）。聚在它周围的邻居全是“一条长烷基链 + 季铵盐阳离子头基”，同时还混进了两个**纯长链烷烃**。

为什么季铵盐表面活性剂（$\log S$ 在 $-3.0 \sim -4.7$，较易溶于水）会和完全不溶于水的纯长链烷烃（$\log S$ 在 $-6.0 \sim -7.6$，溶解度相差三四个数量级）在指纹空间里成为近邻？
- 因为在 2048 位 Morgan 指纹中，长烷基链的 `-CH2-CH2-` 局部环境占据了绝大多数点亮的比特，而真正决定水溶性的**带电荷季铵盐头基 `-N+(CH3)3`** 只改变了链端的少数几个比特！
- 在药物化学与材料信息学中，这种**“分子表征空间距离极近、但目标性质发生剧烈跳变”**的现象被称为**活性悬崖 / 性质悬崖（Activity / Property Cliff）**。它深刻提醒我们：**“两个分子相不相似”完全取决于你采用的表征方式**。

### 6.3 解读 t-SNE 图的四大避坑准则（兼谈 UMAP 与 TMAP）

t-SNE 生成的聚类岛屿图非常美观，但在科研论文中解读 t-SNE 时必须谨记以下四条铁律（因此我们上面特意隐藏了坐标轴刻度 `set_xticks([])`）：

1. **岛屿之间的远近距离没有全局定量意义**：t-SNE 只保证局部近邻聚拢，两个岛在图上隔得很远，并不代表它们在化学上比相邻的两个岛差异更大；
2. **岛屿的膨胀或收缩不代表真实方差大小**：t-SNE 会自适应局部密度，原本稀疏的家族和紧凑的家族在图上可能被拉伸成差不多大小的团块；
3. **拓扑形态依赖于超参数 `perplexity` 与随机种子**：调整 `perplexity` 会改变岛屿分裂与合并的颗粒度；
4. **不支持直接 `transform` 新样本**：与 PCA 不同，标准 t-SNE 是非参数化的坐标优化过程，没有学到一个显式的映射函数 $f(x)$。如果要把新分子画进 t-SNE 图，只能把新老分子拼在一起**重新运行 `fit_transform`**。

> **延伸工具**：近年来在化学空间可视化中，**UMAP**（`umap-learn`）与 **TMAP**（Tree-Map）也很流行。UMAP 计算速度比 t-SNE 更快、对全局拓扑结构的保持更好，而且支持对新分子直接调用 `.transform()`。

### 🏋️ 练习 03a-2：观察 `perplexity` 对 t-SNE 局部与全局颗粒度的影响

`perplexity`（困惑度）控制每个点在构建概率分布时关注的“有效近邻数量”。
1. 为了加快运行速度，先用 `sub_idx = np.random.RandomState(0).choice(len(df), 2000, replace=False)` 随机抽取 2,000 个分子的子集（取出对应的 `sub_df` 和 50 维 PCA 特征 `sub_50 = fp_50[sub_idx]`）；
2. 分别设置 `perplexity=5` 和 `perplexity=100` 运行 `TSNE(n_components=2, init="pca", random_state=42)`，将两张结果图按 `class` 着色并排画出来，对比它们的形态差异。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 练习 03a-3：将有机半导体分子并入指纹空间做 t-SNE 近邻分析

由于 t-SNE 没有 `.transform()` 方法，请把第 4 节中 9 个有机半导体分子（`candidates`）的 50 维指纹 PCA 特征（`pca_fp.transform(osc_fps)`）**拼接**到练习 `03a-2` 的 2,000 个分子子集后面（共 2,009 行），用 `perplexity=30, random_state=42` 一起运行 t-SNE：
1. 将 2,000 个背景分子画成浅灰色散点，9 个有机半导体分子用红点标出并添加文字标注；
2. 观察：`pentacene`、`BTBT`、`rubrene` 是否落在了同一片芳香稠环区域？而带长烷基链的 `3-hexylthiophene` 和 `C8-BTBT` 被拉到了哪里？
3. 找出 t-SNE 图上离 `3-hexylthiophene` 最近的 5 个 AqSolDB 分子，打印它们的名称和 SMILES，体会“分子表征如何决定谁和谁相似”。

In [ ]:
# 在下方填写你的代码：


---

## 小结

| 核心概念 | 方法与代码要点 | 物理化学与数据科学启示 |
|---|---|---|
| **线性降维 PCA** | 先 `StandardScaler`，再 `PCA(n_components=...)` | 17 个描述符高度共线，前 5 个主成分占 >90% 方差；PC1 反映分子大小，PC2 反映亲油 vs 极性（与 $\log S$ 显著相关） |
| **新样本投影与适用域** | 严格复用训练集 `scaler.transform` 和 `pca.transform`；用 `NearestNeighbors` 算全空间距离 | 二维图仅保留部分方差，需在 17 维全空间核查最近邻距离；全局描述符难以区分并五苯与弯曲稠环同分异构体 |
| **高维稀疏指纹降维** | `PCA(50)` 预降维去噪 + `TSNE(perplexity=30)` | 二值指纹维度间缺乏全局线性相关性，直接 PCA 仅保留 9% 方差；t-SNE 能还原局部结构岛屿，但岛间距离无定量意义且不支持直接 `.transform()` |
| **表征决定相似性** | 检视局部近邻分子与性质悬崖（Activity Cliff） | Morgan 指纹被长碳链主导，会把带电季铵盐表面活性剂与疏水烷烃归为近邻，也会把 `C8-BTBT` 拉向长链分子区 |

**下一节预告（`03b_clustering.ipynb`）**：在 t-SNE 图上我们用肉眼看到了一个个分子“岛屿”，能不能不靠肉眼看二维图，直接让算法在高维空间里自动把分子分成若干个簇（Cluster），并利用聚类来构建更严格的模型外推测试集？